<a href="https://colab.research.google.com/github/KimMaynard/Athena/blob/main/finalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#install libraries
!pip install requests
!pip install nltk scikit-learn

In [12]:
import requests
import random
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
def get_full_wikipedia_text(topic):
    topic = topic.replace(" ", "_")  # format for URL
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": True,
        "titles": topic,
        "redirects": 1
    }

    headers = {"User-Agent": "AI-Quiz-Generator/1.0 (student project)"}

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        pages = data["query"]["pages"]

        for page in pages.values():
            if "missing" in page:
                print(f"Error: Topic '{topic}' not found on Wikipedia.")
                return None
            return page.get("extract", "")

        return None

    except requests.exceptions.RequestException as e:
        print("Error fetching data:", e)
        return None

In [14]:
def get_keywords(text, difficulty="easy"):
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text.lower())
    filtered_words = [w for w in words if w not in stop_words]

    clean_text = " ".join(filtered_words)
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([clean_text])

    scores = zip(vectorizer.get_feature_names_out(), tfidf.toarray()[0])
    sorted_words = sorted(scores, key=lambda x: x[1], reverse=True)

    # Difficulty selection
    total = len(sorted_words)
    if difficulty == "easy":
        selected = sorted_words[:max(5, total//3)]
    elif difficulty == "medium":
        selected = sorted_words[total//3 : 2*total//3]
    else:  # hard
        selected = sorted_words[2*total//3 :]

    keywords = [word for word, score in selected]
    return keywords

In [15]:
def get_distractors(keyword, all_keywords):
    distractors = [word for word in all_keywords if word != keyword]
    random.shuffle(distractors)
    return distractors[:3]  # 3 wrong answers

In [16]:
def generate_quiz(text, num_questions=5, difficulty="easy"):
    sentences = text.split(".")  # simple sentence split
    keywords = get_keywords(text, difficulty)

    quiz = []
    used_keywords = set()

    for keyword in keywords:
        if len(quiz) >= num_questions:
            break
        if keyword in used_keywords:
            continue

        # Find a sentence containing the keyword
        sentence = next((s for s in sentences if keyword in s), "")
        if not sentence:
            continue

        question_text = sentence.replace(keyword, "_____")
        distractors = get_distractors(keyword, keywords)
        options = distractors + [keyword]
        random.shuffle(options)

        quiz.append({
            "question": question_text.strip(),
            "options": options,
            "answer": keyword
        })

        used_keywords.add(keyword)

    return quiz

In [17]:
topic = input("Enter a topic: ")
text = get_full_wikipedia_text(topic)

if text:
    text = text[:5000]  # optional: limit size
    quiz = generate_quiz(text, num_questions=5, difficulty="hard")

    for i, q in enumerate(quiz):
        print(f"Question {i+1}: {q['question']}")
        for j, option in enumerate(q['options']):
            print(f"  {chr(65+j)}. {option}")
        print(f"Answer: {q['answer']}\n")
else:
    print("Could not generate quiz.")

Enter a topic: kanji
Question 1: There are nearly 3,000 kanji used in Japanese _____ and in common communication
  A. names
  B. phonographic
  C. 汉字
  D. telephone
Answer: names

Question 2: There are _____ 3,000 kanji used in Japanese names and in common communication
  A. nearly
  B. new
  C. terms
  D. restructure
Answer: nearly

Question 3: After the Meiji Restoration, Japan made its own efforts to simplify the Kanji characters, now known as shinjitai (新字体; '_____ character form'), by a process similar to China's simplification efforts, with the intention to increase literacy among the general public
  A. part
  B. public
  C. new
  D. praised
Answer: new

Question 4: These wooden boards were used for communication between government _____, tags for goods transported between various countries, and the practice of writing
  A. wrote
  B. offices
  C. remain
  D. reading
Answer: offices

Question 5: == History ==

Chinese characters first came to Japan on _____ seals, letters, sword